# Task 1: Universal DAE Hyperparameter Optimization (Optuna)

This notebook executes the **Optuna Hyperparameter Search and Full-Schedule Retraining** for Task 1 as specified in the assignment.

### Mathematical Formulation
The universal denoising autoencoder performs:
$$\hat{x} = D\big(E(\tilde{x})\big)$$
optimizing the composite reconstruction loss:
$$\mathcal{L}_{AE} = \alpha\,\mathcal{L}_{L1}(x,\hat{x}) + (1-\alpha)\big(1-\mathrm{SSIM}(x,\hat{x})\big)$$

### Optuna Search Space Requirements
As required by the assignment, the Optuna study investigates:
1. **Learning Rate** ($\text{lr} \in [10^{-4}, 10^{-3}]$ log scale)
2. **Batch Size** ($B \in \{16, 32, 64\}$)
3. **Bottleneck Dimension** ($\text{latent\_dim} \in \{64, 128, 256\}$)
4. **Encoder Base Channels** ($\text{base\_channels} \in \{32, 64\}$)
5. **Dropout Rate** ($\text{dropout} \in [0.0, 0.3]$)
6. **Loss Weighting Factor** ($\alpha \in [0.5, 0.95]$)

### Optimization Strategy:
- Persistent SQLite storage (`task1_study.db`) with TPE sampler and MedianPruner.
- Reduced-epoch trials (15 epochs) for rapid hyperparameter exploration.
- Winning configuration retrained for the full schedule (60 epochs), saving `task1_universal_ae_final_best.pt`.
- Mandatory **Skip-Connection Ablation** comparing the winning bottleneck model against an equivalent model with a skip connection.

In [ ]:
import os
# 1. Configuration & Optuna Settings
N_TRIALS = 25
OPTUNA_EPOCHS = 15      # Reduced epochs per trial (Plan 7 §Tasks 1 & 2b)
FULL_TRAIN_EPOCHS = 60  # Full schedule retrain for winner (Plan 6 §1.6)

STUDY_NAME = "task1_universal_dae"
STORAGE_DB = "sqlite:///task1_study.db"

USE_WANDB = bool(os.getenv("WANDB_API_KEY"))
WANDB_PROJECT = os.getenv("WANDB_PROJECT", "genai-assignment")
SEED = 42

In [ ]:
# 2. Imports and Environment Setup
import sys
from pathlib import Path
import yaml
import torch
from torch.utils.data import DataLoader
import optuna
from optuna.trial import TrialState
import matplotlib.pyplot as plt

RESEARCH_ROOT = Path("..").resolve()
if str(RESEARCH_ROOT) not in sys.path:
    sys.path.insert(0, str(RESEARCH_ROOT))

from constants import PET_IMAGES_DIR, CHECKPOINTS_MANIFEST, CONFIGS_ROOT
from src.device_utils import get_device, device_report
from src.datasets import PetDataset
from src.models_dae import ConvDAE
from src.train_loop import train_one_epoch_dae, validate_dae, set_seed
from src.checkpoint_utils import create_checkpoint_dict, save_checkpoint

set_seed(SEED)
device = get_device(verbose=True)
print(f"[Optuna] Storage: {STORAGE_DB}, Study: {STUDY_NAME}")

In [ ]:
# 3. Data Loading
train_dataset = PetDataset(mode="train", corruption_mode="all", seed=SEED)
val_dataset = PetDataset(mode="val", corruption_mode="all", seed=SEED)
print(f"[Data] Train: {len(train_dataset)}, Val: {len(val_dataset)}")

In [ ]:
# 4. Optuna Objective
def objective(trial: optuna.Trial) -> float:
    lr = trial.suggest_float("lr", 1e-4, 1e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
    latent_dim = trial.suggest_categorical("latent_dim", [64, 128, 256])
    base_channels = trial.suggest_categorical("base_channels", [32, 64])
    dropout = trial.suggest_float("dropout", 0.0, 0.3)
    alpha = trial.suggest_float("alpha", 0.5, 0.95)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    model = ConvDAE(base_channels=base_channels, latent_dim=latent_dim,
                    dropout=dropout, use_skip=False).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    best_val_loss = float("inf")
    for epoch in range(1, OPTUNA_EPOCHS + 1):
        train_one_epoch_dae(model, train_loader, optimizer, device, alpha=alpha)
        metrics = validate_dae(model, val_loader, device, alpha=alpha)
        best_val_loss = min(best_val_loss, metrics["val_loss"])
        trial.report(metrics["val_loss"], step=epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return best_val_loss


In [ ]:
# 5. Run Optuna Study Optimization
pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3)
study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STORAGE_DB,
    load_if_exists=True,
    direction="minimize",
    pruner=pruner
)

print(f"[Optuna] Starting optimization with {N_TRIALS} trials...")
study.optimize(objective, n_trials=N_TRIALS, timeout=None)

pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

print(f"[Optuna Study Finished]")
print(f"  Total trials: {len(study.trials)}")
print(f"  Complete: {len(complete_trials)}, Pruned: {len(pruned_trials)}")
print(f"  Best trial: #{study.best_trial.number}")
print(f"  Best val loss: {study.best_value:.4f}")
print("  Best parameters:")
for k, v in study.best_params.items():
    print(f"    {k}: {v}")

In [ ]:
# 6. Save Best Config to YAML
best_config_path = CONFIGS_ROOT / "task1_best_config.yaml"
best_config_data = {
    "task": "task1_universal_dae",
    "source_notebook": "task1_dae_optuna.ipynb",
    "best_trial_number": int(study.best_trial.number),
    "best_value": float(study.best_value),
    "params": study.best_params,
}

with open(best_config_path, "w") as f:
    yaml.dump(best_config_data, f, indent=2)

print(f"[Config] Exported best hyperparameters to {best_config_path}")

In [ ]:
# 7. Retrain Winning Model on Full Schedule
best_config_path = CONFIGS_ROOT / "task1_best_config.yaml"
if best_config_path.exists():
    with open(best_config_path, "r") as f:
        saved_config = yaml.safe_load(f) or {}
    p = saved_config.get("params") or study.best_params
    best_trial_number = saved_config.get("best_trial_number")
    if best_trial_number is None:
        best_trial_number = study.best_trial.number
    best_trial_value = saved_config.get("best_value")
    if best_trial_value is None:
        best_trial_value = study.best_value
    print(f"[Config] Loaded best parameters and trial metadata from {best_config_path}")
else:
    p = study.best_params
    best_trial_number = study.best_trial.number
    best_trial_value = study.best_value
    print("[Config] Best-params file not found; using the Optuna study parameters")
batch_size = p["batch_size"]
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
winner_model = ConvDAE(base_channels=p["base_channels"], latent_dim=p["latent_dim"],
                       dropout=p["dropout"], use_skip=False).to(device)
optimizer = torch.optim.Adam(winner_model.parameters(), lr=p["lr"])
best_val_loss = float("inf")
checkpoint_dir = RESEARCH_ROOT / "checkpoints"
for epoch in range(1, FULL_TRAIN_EPOCHS + 1):
    train_metrics = train_one_epoch_dae(winner_model, train_loader, optimizer, device, alpha=p["alpha"])
    val_metrics = validate_dae(winner_model, val_loader, device, alpha=p["alpha"])
    val_loss = val_metrics["val_loss"]
    is_best = val_loss < best_val_loss
    best_val_loss = min(best_val_loss, val_loss)
    ckpt = create_checkpoint_dict(
        winner_model, optimizer, epoch, val_loss,
        optuna_trial_number=best_trial_number, random_seed=SEED,
        extra_metadata={**p, "alpha": p["alpha"], "optuna_best_value": best_trial_value},
    )
    save_checkpoint(
        checkpoint_dir, "task1_universal_ae_final", ckpt, is_best=is_best,
        task_name="task1", phase_name="final", device_name=device_report()["device_name"],
    )
    print(f"Epoch {epoch:02d}/{FULL_TRAIN_EPOCHS}: train={train_metrics['train_loss']:.4f}, "
          f"val={val_loss:.4f} {'*Best*' if is_best else ''}")
print(f"Best validation loss: {best_val_loss:.4f}")


In [ ]:
# 8. Skip-Connection Ablation
skip_model = ConvDAE(base_channels=p["base_channels"], latent_dim=p["latent_dim"],
                     dropout=p["dropout"], use_skip=True).to(device)
skip_optimizer = torch.optim.Adam(skip_model.parameters(), lr=p["lr"])
for epoch in range(1, 16):
    train_one_epoch_dae(skip_model, train_loader, skip_optimizer, device, alpha=p["alpha"])
no_skip_eval = validate_dae(winner_model, val_loader, device, alpha=p["alpha"])
skip_eval = validate_dae(skip_model, val_loader, device, alpha=p["alpha"])
print(f"No skip: val loss={no_skip_eval['val_loss']:.4f}, SSIM={no_skip_eval['val_ssim']:.4f}")
print(f"With skip: val loss={skip_eval['val_loss']:.4f}, SSIM={skip_eval['val_ssim']:.4f}")


## 9. Held-Out Test Images: Final Universal DAE

In [ ]:
# Run deterministic held-out corruption cases through the full-schedule best checkpoint.
from src.checkpoint_utils import load_checkpoint
from src.evaluation import PetBenchmarkDataset, image_quality_metrics

final_test_model = ConvDAE(base_channels=p["base_channels"], latent_dim=p["latent_dim"],
                           dropout=p["dropout"], use_skip=False).to(device)
final_test_path = RESEARCH_ROOT / "checkpoints" / "task1_universal_ae_final_best.pt"
final_test_meta = load_checkpoint(final_test_path, final_test_model, device=device)
final_test_model.eval()
test_dataset = PetBenchmarkDataset()
subset_ids = list(range(min(8, len(test_dataset))))
fig, axes = plt.subplots(len(subset_ids), 3, figsize=(9, 3 * len(subset_ids)), squeeze=False)
metric_rows = []
with torch.inference_mode():
    for row, idx in enumerate(subset_ids):
        corrupted, clean, label, corruption, severity, filename = test_dataset[idx]
        prediction = final_test_model(corrupted.unsqueeze(0).to(device)).cpu()[0]
        scores = image_quality_metrics(prediction.unsqueeze(0), clean.unsqueeze(0))
        metric_rows.append(scores)
        for col, (image, title) in enumerate([(corrupted, f"Input: {corruption}/{severity}"),
                                               (prediction, "Restored"), (clean, "Target")]):
            axes[row, col].imshow(image.permute(1, 2, 0).clamp(0, 1).numpy())
            axes[row, col].set_title(f"{filename} — {title}" if col == 0 else title)
            axes[row, col].axis("off")
plt.tight_layout()
print(f"Tested {len(subset_ids)} held-out cases with checkpoint epoch {final_test_meta.get('epoch')}.")
print("Mean test metrics:", {key: round(sum(m[key] for m in metric_rows) / len(metric_rows), 4)
                               for key in metric_rows[0]})
